# Notebook 03: Deep Pay Equity & Wage Decomposition Analysis

**EquiPay Canada — Research-Grade Labour Economics Analysis**

---

## Overview

This notebook implements **research-grade statistical techniques** to analyze the gender wage gap in Canada, following established methodologies from labour economics literature.

### Research Questions
1. What is the magnitude of the raw and adjusted gender wage gap?
2. How much of the gap is explained vs unexplained (Blinder-Oaxaca)?
3. Does the gap vary across the wage distribution (glass ceiling/floor)?
4. What drives the unexplained component (detailed decomposition)?

## Advanced Methodological Framework

### Decomposition Methods Hierarchy

| Method | Components | Interpretation | Reference |
|--------|------------|----------------|-----------|
| **Twofold B-O** | Explained + Unexplained | Standard approach | Oaxaca (1973) |
| **Threefold B-O** | Endowments + Coefficients + Interaction | More nuanced | Jann (2008) |
| **Detailed Decomposition** | Variable-by-variable contribution | Identify key drivers | Fortin et al. (2011) |
| **Quantile Decomposition** | Gap across distribution | Glass ceiling detection | Machado & Mata (2005) |

### Blinder-Oaxaca Decomposition Theory

**Twofold Decomposition:**
$$\bar{Y}_M - \bar{Y}_F = \underbrace{(\bar{X}_M - \bar{X}_F)\hat{\beta}^*}_{\text{Explained (Endowments)}} + \underbrace{\bar{X}_M(\hat{\beta}_M - \hat{\beta}^*) + \bar{X}_F(\hat{\beta}^* - \hat{\beta}_F)}_{\text{Unexplained (Coefficients)}}$$

**Threefold Decomposition:**
$$\bar{Y}_M - \bar{Y}_F = \underbrace{(\bar{X}_M - \bar{X}_F)\hat{\beta}_F}_{\text{Endowments (E)}} + \underbrace{\bar{X}_F(\hat{\beta}_M - \hat{\beta}_F)}_{\text{Coefficients (C)}} + \underbrace{(\bar{X}_M - \bar{X}_F)(\hat{\beta}_M - \hat{\beta}_F)}_{\text{Interaction (I)}}$$

Where:
- **Endowments (E)**: Gap due to differences in characteristics (education, experience)
- **Coefficients (C)**: Gap due to differential returns (potential discrimination)
- **Interaction (I)**: Simultaneous difference in endowments and coefficients

### Detailed Decomposition

Following Fortin, Lemieux & Firpo (2011), we report the **contribution of each variable** to both explained and unexplained components:

$$\text{Explained}_j = (\bar{X}_{Mj} - \bar{X}_{Fj})\hat{\beta}^*_j$$

This reveals which characteristics (e.g., occupation vs education) drive the gap.

### Inference Methods

| Method | Purpose |
|--------|---------|
| **Bootstrap Standard Errors** | Uncertainty for decomposition estimates |
| **Confidence Intervals** | Report ranges, not just point estimates |
| **Robust SE (HC3)** | Heteroskedasticity-consistent inference |

## Key References
- Blinder, A. S. (1973). Wage discrimination: Reduced form and structural estimates. *Journal of Human Resources*, 8(4), 436-455.
- Fortin, N., Lemieux, T., & Firpo, S. (2011). Decomposition methods in economics. *Handbook of Labor Economics*, 4, 1-102.
- Jann, B. (2008). The Blinder-Oaxaca decomposition for linear regression models. *The Stata Journal*, 8(4), 453-479.
- Machado, J. A., & Mata, J. (2005). Counterfactual decomposition of changes in wage distributions. *JASA*, 100(472), 1053-1062.
- Oaxaca, R. (1973). Male-female wage differentials in urban labor markets. *International Economic Review*, 14(3), 693-709.

## Data Source
- **Labour Force Survey PUMF** (Statistics Canada Catalogue 71M0001X)
- **Period**: 2010-2025 (N ≈ 100,000+ observations)

---

In [ ]:
# ============================================================================
# SETUP: Import Libraries and Configure Environment
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Add project root
sys.path.insert(0, str(Path.cwd().parent))

# Import project modules
from src.constants import (
    COLS, GENDER_CODES, EDUCATION_CODES, NOC_10_CODES, PROVINCE_CODES,
    normalize_column_names, DATA_SCOPE_START, DATA_SCOPE_END
)
from src.analysis import PayEquityAnalyzer
from src.macro_data import MACRO_DATA, get_macro_dataframe, ECONOMIC_PERIODS

# Publication-quality figures
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white'
})
plt.style.use('seaborn-v0_8-whitegrid')

# Ensure figures directory exists
Path('../reports/figures').mkdir(parents=True, exist_ok=True)

print("✓ Libraries loaded successfully")
print(f"✓ Analysis period: {DATA_SCOPE_START}-{DATA_SCOPE_END}")

## 1. Data Loading & Validation

Load the processed LFS data and verify sample composition.

In [ ]:
# Load processed data
data_path = Path('../data/processed/lfs_processed.csv')

if not data_path.exists():
    print("⚠️ Run notebook 01 first to generate data!")
    raise FileNotFoundError(f"{data_path} not found")

df = pd.read_csv(data_path)
df = normalize_column_names(df)

# Identify columns
gender_col = COLS.GENDER if COLS.GENDER in df.columns else 'SEX'

# Use REAL hourly earnings for pay equity analysis (inflation-adjusted)
# This ensures consistent comparison across years (2010-2025)
if COLS.REAL_HOURLY_EARNINGS in df.columns:
    wage_col = COLS.REAL_HOURLY_EARNINGS
    print("✓ Using REAL hourly earnings (inflation-adjusted to 2010$)")
elif 'REAL_HRLYEARN' in df.columns:
    wage_col = 'REAL_HRLYEARN'
    print("✓ Using REAL hourly earnings (inflation-adjusted to 2010$)")
else:
    wage_col = COLS.HOURLY_EARNINGS
    print("⚠ Real wages not available - using nominal wages")

print(f"✓ Dataset: {len(df):,} records")
print(f"\nGender distribution:")
print(f"  Male: {(df[gender_col] == 1).sum():,} ({(df[gender_col] == 1).mean()*100:.1f}%)")
print(f"  Female: {(df[gender_col] == 2).sum():,} ({(df[gender_col] == 2).mean()*100:.1f}%)")

## 2. Raw Wage Gap Analysis

**Unadjusted comparison** of mean and median wages by gender, with:
- Two-sample t-test (unequal variances)
- Cohen's d effect size
- Mann-Whitney U test (non-parametric alternative)

In [ ]:
# Initialize analyzer
analyzer = PayEquityAnalyzer(df)

# Compute raw wage gap
raw_gap = analyzer.compute_raw_wage_gap()

# Display results
print("=" * 60)
print("RAW GENDER WAGE GAP ANALYSIS")
print("=" * 60)

print(f"\nMale Workers:")
print(f"  Mean hourly wage: ${raw_gap['male']['mean']:.2f}")
print(f"  Median hourly wage: ${raw_gap['male']['median']:.2f}")
print(f"  Sample size: {raw_gap['male']['n']:,}")

print(f"\nFemale Workers:")
print(f"  Mean hourly wage: ${raw_gap['female']['mean']:.2f}")
print(f"  Median hourly wage: ${raw_gap['female']['median']:.2f}")
print(f"  Sample size: {raw_gap['female']['n']:,}")

print(f"\n" + "-" * 60)
print(f"RAW WAGE GAP: {raw_gap['raw_gap']['mean_gap_pct']:.1f}%")
print(f"Women earn ${raw_gap['raw_gap']['female_to_male_ratio']:.2f} for every $1.00 men earn")
print(f"Dollar difference: ${raw_gap['raw_gap']['mean_gap']:.2f}/hour")

In [ ]:
# Statistical significance
print("\nStatistical Significance:")
print("-" * 40)
t_test = raw_gap['statistical_tests']['t_test']
print(f"T-statistic: {t_test['statistic']:.2f}")
print(f"P-value: {t_test['p_value']:.2e}")
print(f"Significant at α=0.05: {'Yes ✓' if t_test['significant_05'] else 'No'}")
print(f"Significant at α=0.01: {'Yes ✓' if t_test['significant_01'] else 'No'}")

effect = raw_gap['statistical_tests']['effect_size']
print(f"\nEffect Size (Cohen's d): {effect['cohens_d']:.3f} ({effect['interpretation']})")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Bar comparison
ax = axes[0]
genders = ['Male', 'Female']
means = [raw_gap['male']['mean'], raw_gap['female']['mean']]
colors = ['#3498db', '#e74c3c']
bars = ax.bar(genders, means, color=colors)
ax.bar_label(bars, fmt='$%.2f')
ax.set_ylabel('Average Hourly Wage ($)')
ax.set_title('Average Wage by Gender')

# Box plots
ax = axes[1]
df['Gender_Label'] = df[gender_col].map({1: 'Male', 2: 'Female'})
df.boxplot(column=wage_col, by='Gender_Label', ax=ax)
ax.set_title('Wage Distribution by Gender')
ax.set_ylabel('Hourly Wage ($)')
ax.set_xlabel('')
plt.suptitle('')

# Gap visualization
ax = axes[2]
ax.barh(['Gender\nWage Gap'], [raw_gap['raw_gap']['mean_gap_pct']], color='crimson', height=0.5)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Wage Gap (%)')
ax.set_title(f"Raw Gender Wage Gap: {raw_gap['raw_gap']['mean_gap_pct']:.1f}%")
ax.set_xlim(0, max(25, raw_gap['raw_gap']['mean_gap_pct'] + 5))

plt.tight_layout()
plt.show()

## 3. Adjusted Wage Gap (Mincer Regression)

**Controls for observable characteristics:**
- Education level (EDUC)
- Experience proxy (AGE_12)
- Occupation (NOC_10)
- Province (PROV)
- Employment type (FTPTMAIN)

The residual gender effect represents the **unexplained gap** after accounting for human capital differences.

In [ ]:
# Identify available control variables
potential_controls = [COLS.EDUC, COLS.NOC_10, COLS.AGE_12, COLS.PROV, COLS.FTPTMAIN, COLS.TENURE, 'EXPERIENCE']
control_vars = [c for c in potential_controls if c in df.columns and df[c].dtype in ['int64', 'float64', 'int32', 'float32']]

print(f"Control variables: {control_vars}")

if control_vars:
    adjusted_gap = analyzer.compute_adjusted_wage_gap(control_vars)
    
    print("\n" + "=" * 60)
    print("ADJUSTED WAGE GAP ANALYSIS")
    print("=" * 60)
    
    print(f"\nUnadjusted Model (gender only):")
    print(f"  Wage gap: {adjusted_gap['unadjusted_model']['gap_pct']:.1f}%")
    print(f"  R²: {adjusted_gap['unadjusted_model']['r_squared']:.4f}")
    
    print(f"\nAdjusted Model (with controls):")
    print(f"  Wage gap: {adjusted_gap['adjusted_model']['gap_pct']:.1f}%")
    print(f"  R²: {adjusted_gap['adjusted_model']['r_squared']:.4f}")
    print(f"  P-value: {adjusted_gap['adjusted_model']['p_value']:.4f}")
    
    print(f"\n" + "-" * 60)
    print(f"Gap explained by controls: {adjusted_gap['gap_reduction']['absolute']:.1f} percentage points")
    print(f"Unexplained gap: {adjusted_gap['interpretation']['unexplained_gap']:.1f}%")
else:
    print("⚠️ Insufficient control variables for adjusted analysis.")

In [ ]:
# Gap comparison visualization
if 'adjusted_gap' in dir():
    fig, ax = plt.subplots(figsize=(10, 5))
    
    gaps = [
        abs(adjusted_gap['unadjusted_model']['gap_pct']),
        abs(adjusted_gap['adjusted_model']['gap_pct'])
    ]
    labels = ['Unadjusted\n(Raw Gap)', 'Adjusted\n(With Controls)']
    colors = ['#d62728', '#2ca02c']
    
    bars = ax.bar(labels, gaps, color=colors, width=0.5)
    ax.bar_label(bars, fmt='%.1f%%')
    ax.set_ylabel('Wage Gap (%)')
    ax.set_title('Raw vs Adjusted Gender Wage Gap')
    
    # Add annotation
    reduction = gaps[0] - gaps[1]
    ax.annotate(f'Reduction: {reduction:.1f} pp', 
                xy=(0.5, max(gaps)/2), fontsize=12,
                ha='center', color='gray')
    
    plt.tight_layout()
    plt.show()

## 4. Blinder-Oaxaca Decomposition

The gold standard for wage gap analysis (Blinder, 1973; Oaxaca, 1973). 

**Interpretation:**
- **Explained (Endowments)**: Gap due to differences in characteristics
- **Unexplained (Coefficients)**: Gap due to differential returns — often interpreted as discrimination, but may also reflect omitted variables

In [ ]:
# Perform decomposition
if control_vars:
    decomp = analyzer.oaxaca_blinder_decomposition(control_vars)
    
    print("=" * 60)
    print("OAXACA-BLINDER WAGE DECOMPOSITION")
    print("=" * 60)
    
    print(f"\nTotal Wage Gap: {decomp['total_gap']['percentage']:.1f}%")
    
    print(f"\nExplained Component: {decomp['explained']['pct_of_total']:.1f}% of gap")
    print(f"  → {decomp['explained']['interpretation']}")
    
    print(f"\nUnexplained Component: {decomp['unexplained']['pct_of_total']:.1f}% of gap")
    print(f"  → {decomp['unexplained']['interpretation']}")
    
    print(f"\nSample sizes: Male={decomp['sample_sizes']['male']:,}, Female={decomp['sample_sizes']['female']:,}")

In [ ]:
# Decomposition visualization
if 'decomp' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Pie chart
    ax = axes[0]
    sizes = [abs(decomp['explained']['pct_of_total']), 
             abs(decomp['unexplained']['pct_of_total'])]
    labels = ['Explained\n(Characteristics)', 'Unexplained\n(Potential Discrimination)']
    colors = ['#2ca02c', '#d62728']
    explode = (0, 0.05)
    
    ax.pie(sizes, explode=explode, labels=labels, colors=colors,
           autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
    ax.set_title(f'Wage Gap Decomposition\n(Total Gap: {decomp["total_gap"]["percentage"]:.1f}%)')
    
    # Bar chart
    ax = axes[1]
    components = ['Total Gap', 'Explained', 'Unexplained']
    # Use 'value' key instead of 'percentage' for explained/unexplained
    total_gap_pct = decomp['total_gap']['percentage']
    explained_pct = decomp['explained'].get('percentage', decomp['explained']['pct_of_total'] * total_gap_pct / 100)
    unexplained_pct = decomp['unexplained'].get('percentage', decomp['unexplained']['pct_of_total'] * total_gap_pct / 100)
    values = [total_gap_pct, explained_pct, unexplained_pct]
    colors = ['gray', '#2ca02c', '#d62728']
    
    bars = ax.bar(components, values, color=colors)
    ax.bar_label(bars, fmt='%.1f%%')
    ax.set_ylabel('Percentage Points')
    ax.set_title('Wage Gap Components')
    ax.axhline(0, color='black', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()

## 4.2 Threefold Decomposition (Jann, 2008)

The **threefold decomposition** provides a more nuanced breakdown by separating the interaction term:

$$\Delta\bar{Y} = \underbrace{(\bar{X}_M - \bar{X}_F)\hat{\beta}_F}_{\text{Endowments (E)}} + \underbrace{\bar{X}_F(\hat{\beta}_M - \hat{\beta}_F)}_{\text{Coefficients (C)}} + \underbrace{(\bar{X}_M - \bar{X}_F)(\hat{\beta}_M - \hat{\beta}_F)}_{\text{Interaction (I)}}$$

| Component | Interpretation | Policy Implication |
|-----------|----------------|-------------------|
| **Endowments (E)** | Differences in characteristics evaluated at female coefficients | Human capital investment |
| **Coefficients (C)** | Differences in returns evaluated at female characteristics | Anti-discrimination policy |
| **Interaction (I)** | Simultaneous difference in both | Complex structural factors |

In [ ]:
# ============================================================================
# THREEFOLD BLINDER-OAXACA DECOMPOSITION
# ============================================================================
# Jann (2008): The Blinder-Oaxaca decomposition for linear regression models
# ============================================================================

print("=" * 70)
print("THREEFOLD BLINDER-OAXACA DECOMPOSITION")
print("=" * 70)

def threefold_oaxaca_decomposition(df, y_col, gender_col, control_vars, log_transform=True):
    """
    Perform threefold Oaxaca-Blinder decomposition.
    
    Components:
    - E (Endowments): Difference in characteristics at female coefficients
    - C (Coefficients): Difference in returns at female characteristics
    - I (Interaction): Simultaneous difference in both
    
    Returns
    -------
    dict : Decomposition results with E, C, I components
    """
    # Separate by gender
    male_df = df[df[gender_col] == 1].copy()
    female_df = df[df[gender_col] == 2].copy()
    
    # Prepare data
    X_vars = [c for c in control_vars if c != gender_col and c in df.columns]
    
    male_clean = male_df[X_vars + [y_col]].dropna()
    female_clean = female_df[X_vars + [y_col]].dropna()
    
    X_m = male_clean[X_vars].values
    X_f = female_clean[X_vars].values
    
    if log_transform:
        y_m = np.log(male_clean[y_col].clip(lower=1)).values
        y_f = np.log(female_clean[y_col].clip(lower=1)).values
    else:
        y_m = male_clean[y_col].values
        y_f = female_clean[y_col].values
    
    # Add constant
    X_m = sm.add_constant(X_m)
    X_f = sm.add_constant(X_f)
    
    # Fit separate regressions
    model_m = sm.OLS(y_m, X_m).fit()
    model_f = sm.OLS(y_f, X_f).fit()
    
    # Mean characteristics (including constant)
    X_m_bar = X_m.mean(axis=0)
    X_f_bar = X_f.mean(axis=0)
    
    # Coefficients
    beta_m = model_m.params
    beta_f = model_f.params
    
    # Mean outcomes
    y_m_bar = y_m.mean()
    y_f_bar = y_f.mean()
    total_gap = y_m_bar - y_f_bar
    
    # Threefold decomposition
    # E = (X_m - X_f) * beta_f
    E = np.sum((X_m_bar - X_f_bar) * beta_f)
    
    # C = X_f * (beta_m - beta_f)
    C = np.sum(X_f_bar * (beta_m - beta_f))
    
    # I = (X_m - X_f) * (beta_m - beta_f)
    I = np.sum((X_m_bar - X_f_bar) * (beta_m - beta_f))
    
    # Verify: E + C + I should equal total gap
    check_sum = E + C + I
    
    return {
        'total_gap': {'log': total_gap, 'percentage': (np.exp(total_gap) - 1) * 100},
        'endowments': {
            'value': E,
            'pct_of_gap': (E / total_gap) * 100 if total_gap != 0 else 0,
            'wage_effect': (np.exp(E) - 1) * 100
        },
        'coefficients': {
            'value': C,
            'pct_of_gap': (C / total_gap) * 100 if total_gap != 0 else 0,
            'wage_effect': (np.exp(C) - 1) * 100
        },
        'interaction': {
            'value': I,
            'pct_of_gap': (I / total_gap) * 100 if total_gap != 0 else 0,
            'wage_effect': (np.exp(I) - 1) * 100
        },
        'check_sum': check_sum,
        'n_male': len(male_clean),
        'n_female': len(female_clean)
    }

# Perform threefold decomposition
if control_vars:
    threefold = threefold_oaxaca_decomposition(
        df, wage_col, gender_col, control_vars, log_transform=True
    )
    
    print(f"\nTotal Wage Gap: {threefold['total_gap']['percentage']:.2f}%")
    print(f"  (log difference: {threefold['total_gap']['log']:.4f})")
    
    print(f"\nComponent Breakdown:")
    print("-" * 60)
    print(f"{'Component':<20} {'Value':>10} {'% of Gap':>12} {'Wage Effect':>12}")
    print("-" * 60)
    
    print(f"{'Endowments (E)':<20} {threefold['endowments']['value']:>10.4f} "
          f"{threefold['endowments']['pct_of_gap']:>11.1f}% "
          f"{threefold['endowments']['wage_effect']:>11.2f}%")
    
    print(f"{'Coefficients (C)':<20} {threefold['coefficients']['value']:>10.4f} "
          f"{threefold['coefficients']['pct_of_gap']:>11.1f}% "
          f"{threefold['coefficients']['wage_effect']:>11.2f}%")
    
    print(f"{'Interaction (I)':<20} {threefold['interaction']['value']:>10.4f} "
          f"{threefold['interaction']['pct_of_gap']:>11.1f}% "
          f"{threefold['interaction']['wage_effect']:>11.2f}%")
    
    print("-" * 60)
    print(f"{'Sum (E+C+I)':<20} {threefold['check_sum']:>10.4f}")
    
    print(f"\nInterpretation:")
    print(f"  • Endowments (E): {abs(threefold['endowments']['pct_of_gap']):.1f}% due to characteristic differences")
    print(f"  • Coefficients (C): {abs(threefold['coefficients']['pct_of_gap']):.1f}% due to differential returns")
    print(f"  • Interaction (I): {abs(threefold['interaction']['pct_of_gap']):.1f}% due to combined effects")

## 4.3 Detailed Decomposition: Variable-by-Variable Contribution

Following Fortin, Lemieux & Firpo (2011), we decompose the **explained component** into contributions from each variable. This reveals which characteristics (education, occupation, etc.) drive the gender gap.

In [ ]:
# ============================================================================
# DETAILED DECOMPOSITION: Variable-by-Variable Contribution
# ============================================================================
# Fortin, Lemieux & Firpo (2011): Decomposition methods in economics
# ============================================================================

print("=" * 70)
print("DETAILED DECOMPOSITION (Variable-by-Variable)")
print("=" * 70)

def detailed_decomposition(df, y_col, gender_col, control_vars, log_transform=True):
    """
    Compute detailed decomposition showing each variable's contribution.
    
    Returns contribution of each variable to both explained and unexplained.
    """
    # Separate by gender
    male_df = df[df[gender_col] == 1].copy()
    female_df = df[df[gender_col] == 2].copy()
    
    # Prepare data
    X_vars = [c for c in control_vars if c != gender_col and c in df.columns]
    
    male_clean = male_df[X_vars + [y_col]].dropna()
    female_clean = female_df[X_vars + [y_col]].dropna()
    
    # Create DataFrames with constants - reset index to avoid alignment issues
    X_m_df = pd.DataFrame(male_clean[X_vars]).reset_index(drop=True)
    X_f_df = pd.DataFrame(female_clean[X_vars]).reset_index(drop=True)
    
    if log_transform:
        y_m = np.log(male_clean[y_col].clip(lower=1)).reset_index(drop=True)
        y_f = np.log(female_clean[y_col].clip(lower=1)).reset_index(drop=True)
    else:
        y_m = male_clean[y_col].reset_index(drop=True)
        y_f = female_clean[y_col].reset_index(drop=True)
    
    # Add constant
    X_m_const = sm.add_constant(X_m_df)
    X_f_const = sm.add_constant(X_f_df)
    
    # Fit models
    model_m = sm.OLS(y_m, X_m_const).fit()
    model_f = sm.OLS(y_f, X_f_const).fit()
    
    # Pooled model (for reference coefficients)
    X_pooled = sm.add_constant(pd.concat([X_m_df, X_f_df], ignore_index=True))
    y_pooled = pd.concat([y_m.reset_index(drop=True), y_f.reset_index(drop=True)], ignore_index=True)
    model_pooled = sm.OLS(y_pooled, X_pooled).fit()
    
    # Mean characteristics
    X_m_mean = X_m_const.mean()
    X_f_mean = X_f_const.mean()
    
    # Detailed contributions (excluding constant)
    contributions = []
    for var in X_vars:
        # Explained: (X_m - X_f) * beta_pooled
        explained_j = (X_m_mean[var] - X_f_mean[var]) * model_pooled.params[var]
        
        # Unexplained for male: X_m * (beta_m - beta_pooled)
        unexplained_m_j = X_m_mean[var] * (model_m.params[var] - model_pooled.params[var])
        
        # Unexplained for female: X_f * (beta_pooled - beta_f)
        unexplained_f_j = X_f_mean[var] * (model_pooled.params[var] - model_f.params[var])
        
        contributions.append({
            'variable': var,
            'mean_male': X_m_mean[var],
            'mean_female': X_f_mean[var],
            'diff': X_m_mean[var] - X_f_mean[var],
            'beta_pooled': model_pooled.params[var],
            'explained': explained_j,
            'unexplained': unexplained_m_j + unexplained_f_j
        })
    
    return pd.DataFrame(contributions)

# Compute detailed decomposition
if control_vars:
    detailed_df = detailed_decomposition(df, wage_col, gender_col, control_vars)
    
    # Sort by absolute explained contribution
    detailed_df['abs_explained'] = detailed_df['explained'].abs()
    detailed_df = detailed_df.sort_values('abs_explained', ascending=False)
    
    # Calculate total
    total_explained = detailed_df['explained'].sum()
    
    print(f"\nContribution to Explained Component:")
    print("-" * 70)
    print(f"{'Variable':<20} {'Mean M':>10} {'Mean F':>10} {'Diff':>8} {'Contrib.':>10} {'% of Expl.':>10}")
    print("-" * 70)
    
    for _, row in detailed_df.iterrows():
        pct_of_explained = (row['explained'] / total_explained) * 100 if total_explained != 0 else 0
        print(f"{row['variable']:<20} {row['mean_male']:>10.3f} {row['mean_female']:>10.3f} "
              f"{row['diff']:>8.3f} {row['explained']:>10.4f} {pct_of_explained:>9.1f}%")
    
    print("-" * 70)
    print(f"{'TOTAL':<20} {'':<21} {total_explained:>10.4f} {'100.0%':>10}")
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Explained contributions bar chart
    ax = axes[0]
    colors = ['#2ca02c' if x > 0 else '#d62728' for x in detailed_df['explained']]
    ax.barh(detailed_df['variable'], detailed_df['explained'], color=colors)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Contribution to Explained Component (log wage)')
    ax.set_title('Variable Contributions to Explained Gap')
    
    # Pie chart of top contributors
    ax = axes[1]
    top_5 = detailed_df.head(5).copy()
    top_5['pct'] = (top_5['explained'].abs() / top_5['explained'].abs().sum()) * 100
    
    colors_pie = plt.get_cmap('tab10')(range(len(top_5)))
    ax.pie(top_5['pct'], labels=top_5['variable'], autopct='%1.1f%%', 
           colors=colors_pie, startangle=90)
    ax.set_title('Top 5 Contributors (by magnitude)')
    
    plt.tight_layout()
    plt.savefig('../reports/figures/detailed_decomposition.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n📊 Detailed decomposition saved to reports/figures/detailed_decomposition.png")

## 4.4 Intersectionality Analysis

**Intersectionality** (Crenshaw, 1989) recognizes that gender effects are not uniform across subgroups. We examine how the wage gap varies by:
- Gender × Education Level
- Gender × Occupation
- Gender × Province

This reveals whether certain groups face compounded disadvantages.

In [ ]:
# ============================================================================
# INTERSECTIONALITY ANALYSIS
# ============================================================================
# Examining how gender effects vary across subgroups
# ============================================================================

print("=" * 70)
print("INTERSECTIONALITY ANALYSIS")
print("=" * 70)

def intersectional_gap_analysis(df, wage_col, gender_col, groupby_col, labels_map=None):
    """
    Compute gender wage gap within each subgroup.
    
    Returns DataFrame with gap by subgroup.
    """
    results = []
    
    for group_val in df[groupby_col].dropna().unique():
        group_df = df[df[groupby_col] == group_val]
        
        male_wages = group_df[group_df[gender_col] == 1][wage_col].dropna()
        female_wages = group_df[group_df[gender_col] == 2][wage_col].dropna()
        
        if len(male_wages) >= 30 and len(female_wages) >= 30:
            gap_pct = (male_wages.mean() - female_wages.mean()) / male_wages.mean() * 100
            
            # Cohen's d
            pooled_std = np.sqrt(((len(male_wages)-1)*male_wages.std()**2 + 
                                   (len(female_wages)-1)*female_wages.std()**2) / 
                                  (len(male_wages) + len(female_wages) - 2))
            cohens_d = (male_wages.mean() - female_wages.mean()) / pooled_std if pooled_std > 0 else 0
            
            results.append({
                'group': labels_map.get(group_val, str(group_val)) if labels_map else str(group_val),
                'group_val': group_val,
                'n_male': len(male_wages),
                'n_female': len(female_wages),
                'male_mean': male_wages.mean(),
                'female_mean': female_wages.mean(),
                'gap_pct': gap_pct,
                'cohens_d': cohens_d
            })
    
    return pd.DataFrame(results)

# Intersectionality by education
if COLS.EDUC in df.columns:
    intersect_educ = intersectional_gap_analysis(
        df, wage_col, gender_col, COLS.EDUC, EDUCATION_CODES
    )
    
    if len(intersect_educ) > 0:
        print(f"\nGender Wage Gap by Education Level:")
        print("-" * 70)
        print(f"{'Education':<30} {'Gap %':>8} {'Cohen d':>10} {'n_M':>8} {'n_F':>8}")
        print("-" * 70)
        
        for _, row in intersect_educ.sort_values('gap_pct', ascending=False).iterrows():
            print(f"{row['group'][:28]:<30} {row['gap_pct']:>7.1f}% "
                  f"{row['cohens_d']:>10.3f} {row['n_male']:>8,} {row['n_female']:>8,}")
        
        # Visualization
        fig, ax = plt.subplots(figsize=(12, 6))
        
        intersect_sorted = intersect_educ.sort_values('gap_pct', ascending=True)
        colors = ['#d62728' if g > 15 else '#ffc107' if g > 10 else '#2ca02c' 
                  for g in intersect_sorted['gap_pct']]
        
        ax.barh(intersect_sorted['group'].astype(str), intersect_sorted['gap_pct'], 
                color=colors, edgecolor='black', linewidth=0.5)
        ax.axvline(0, color='black', linewidth=0.5)
        ax.set_xlabel('Wage Gap (%)')
        ax.set_title('Gender Wage Gap by Education Level (Intersectionality)')
        
        # Add overall mean line
        overall_gap = intersect_educ['gap_pct'].mean()
        ax.axvline(overall_gap, color='blue', linestyle='--', 
                   label=f'Mean: {overall_gap:.1f}%')
        ax.legend()
        
        plt.tight_layout()
        plt.savefig('../reports/figures/intersectionality_education.png', dpi=150, bbox_inches='tight')
        plt.show()

# Intersectionality by occupation
if COLS.NOC_10 in df.columns:
    intersect_occ = intersectional_gap_analysis(
        df, wage_col, gender_col, COLS.NOC_10, NOC_10_CODES
    )
    
    if len(intersect_occ) > 0:
        print(f"\nGender Wage Gap by Occupation:")
        print("-" * 70)
        
        for _, row in intersect_occ.sort_values('gap_pct', ascending=False).iterrows():
            print(f"{row['group'][:28]:<30} {row['gap_pct']:>7.1f}% "
                  f"{row['cohens_d']:>10.3f}")
        
        # Identify most and least equitable occupations
        if len(intersect_occ) > 1:
            max_gap = intersect_occ.loc[intersect_occ['gap_pct'].idxmax()]
            min_gap = intersect_occ.loc[intersect_occ['gap_pct'].idxmin()]
            
            print(f"\n📊 Key Findings:")
            print(f"  • Largest gap: {max_gap['group']} ({max_gap['gap_pct']:.1f}%)")
            print(f"  • Smallest gap: {min_gap['group']} ({min_gap['gap_pct']:.1f}%)")
            print(f"  • Range: {max_gap['gap_pct'] - min_gap['gap_pct']:.1f} percentage points")

## 5. Macroeconomic Context

Wages are influenced by business cycle conditions. We incorporate macro controls to assess whether the gap varies with economic conditions.

In [ ]:
# Display macro context
print("=" * 60)
print("MACROECONOMIC CONTEXT")
print("=" * 60)

macro_df = get_macro_dataframe()
print("\nCanadian Economic Indicators (2010-2025):")
display(macro_df[['year', 'cpi', 'gdp_growth', 'unemployment']].round(2))

print("\nEconomic Periods:")
for period, (start, end) in ECONOMIC_PERIODS.items():
    print(f"  {period.replace('_', ' ').title()}: {start}-{end}")

In [ ]:
# Macro-adjusted analysis (if YEAR column available)
if 'YEAR' in df.columns or 'SURVYEAR' in df.columns:
    year_col = 'YEAR' if 'YEAR' in df.columns else 'SURVYEAR'
    
    # Merge macro data
    df_macro = df.merge(
        macro_df[['year', 'cpi', 'gdp_growth', 'unemployment']],
        left_on=year_col, right_on='year', how='left'
    )
    
    # Add macro controls
    macro_controls = control_vars + ['cpi', 'gdp_growth', 'unemployment']
    macro_controls = [c for c in macro_controls if c in df_macro.columns]
    
    # Re-run with macro controls
    analyzer_macro = PayEquityAnalyzer(df_macro)
    macro_gap = analyzer_macro.compute_adjusted_wage_gap(macro_controls)
    
    print("\n" + "=" * 60)
    print("MACRO-ADJUSTED WAGE GAP")
    print("=" * 60)
    print(f"Gap with macro controls: {macro_gap['adjusted_model']['gap_pct']:.1f}%")
    print(f"R²: {macro_gap['adjusted_model']['r_squared']:.4f}")
else:
    print("\n⚠️ Year column not available for macro-adjusted analysis")

## 6. Heterogeneity Analysis: Gap by Demographic Groups

**Intersectionality matters.** The wage gap varies by:
- Education level
- Occupation category
- Province/region

In [ ]:
# Gap by education
if COLS.EDUC in df.columns:
    gap_by_educ = analyzer.analyze_by_group(COLS.EDUC)
    gap_by_educ['Education'] = gap_by_educ[COLS.EDUC].map(EDUCATION_CODES)
    
    print("Gender Wage Gap by Education Level:")
    display(gap_by_educ[['Education', 'male_mean', 'female_mean', 'gap_pct', 'significant']].round(2))

In [ ]:
# Visualize gap by education
if 'gap_by_educ' in dir() and len(gap_by_educ) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    gap_clean = gap_by_educ.dropna(subset=['Education']).copy()
    gap_sorted = gap_clean.sort_values('gap_pct', ascending=True)
    
    colors = ['#d62728' if g > 10 else '#ffc107' if g > 5 else '#2ca02c' 
              for g in gap_sorted['gap_pct']]
    
    ax.barh(gap_sorted['Education'].astype(str), gap_sorted['gap_pct'], color=colors)
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.set_xlabel('Wage Gap (%)')
    ax.set_title('Gender Wage Gap by Education Level')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Gap by occupation
if COLS.NOC_10 in df.columns:
    gap_by_occ = analyzer.analyze_by_group(COLS.NOC_10)
    gap_by_occ['Occupation'] = gap_by_occ[COLS.NOC_10].map(NOC_10_CODES)
    
    print("\nGender Wage Gap by Occupation:")
    display(gap_by_occ[['Occupation', 'male_mean', 'female_mean', 'gap_pct', 'significant']].round(2))

In [ ]:
# Visualize gap by occupation
if 'gap_by_occ' in dir() and len(gap_by_occ) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    gap_clean = gap_by_occ.dropna(subset=['Occupation']).copy()
    gap_sorted = gap_clean.sort_values('gap_pct', ascending=True)
    
    colors = ['#d62728' if g > 15 else '#ffc107' if g > 10 else '#2ca02c' 
              for g in gap_sorted['gap_pct']]
    
    ax.barh(gap_sorted['Occupation'].astype(str), gap_sorted['gap_pct'], color=colors)
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.set_xlabel('Wage Gap (%)')
    ax.set_title('Gender Wage Gap by Occupation Category')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Gap by province
if COLS.PROV in df.columns:
    gap_by_prov = analyzer.analyze_by_group(COLS.PROV)
    gap_by_prov['Province'] = gap_by_prov[COLS.PROV].map(PROVINCE_CODES)
    
    print("\nGender Wage Gap by Province:")
    display(gap_by_prov[['Province', 'male_mean', 'female_mean', 'gap_pct', 'significant']].round(2))

## 7. Quantile Analysis: Glass Ceiling & Sticky Floor

**Beyond mean comparisons:** We analyze whether the gap differs across the wage distribution.

- **Glass Ceiling**: Larger gap at high wages (women blocked from top positions)
- **Sticky Floor**: Larger gap at low wages (women stuck in low-paying roles)

In [ ]:
# Quantile analysis
quantile_results = analyzer.quantile_analysis()

print("=" * 60)
print("WAGE GAP ACROSS WAGE DISTRIBUTION")
print("=" * 60)

for percentile, data in quantile_results.items():
    if percentile.startswith('p'):
        print(f"\n{percentile.upper()}:")
        print(f"  Male: ${data['male']:.2f}")
        print(f"  Female: ${data['female']:.2f}")
        print(f"  Gap: {data['gap_pct']:.1f}%")

print(f"\n" + "-" * 60)
ceiling = quantile_results['glass_ceiling_effect']
print(f"Glass Ceiling Effect: {ceiling['interpretation']}")
print(f"Gap difference (P90 vs P10): {ceiling['difference']:.1f} percentage points")

In [ ]:
# Visualize quantile gaps
percentiles = ['p10', 'p25', 'p50', 'p75', 'p90']
gaps = [quantile_results[p]['gap_pct'] for p in percentiles]

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(percentiles, gaps, marker='o', linewidth=2, markersize=10, color='steelblue')
ax.fill_between(percentiles, gaps, alpha=0.3, color='steelblue')
ax.set_xlabel('Wage Percentile')
ax.set_ylabel('Wage Gap (%)')
ax.set_title('Gender Wage Gap Across Wage Distribution')
ax.set_xticklabels(['10th', '25th', '50th', '75th', '90th'])

for i, (p, g) in enumerate(zip(percentiles, gaps)):
    ax.annotate(f'{g:.1f}%', (p, g), textcoords='offset points', 
                xytext=(0, 10), ha='center')

plt.tight_layout()
plt.show()

## 8. Summary Report & Policy Implications

In [ ]:
# Generate summary report
summary = analyzer.generate_summary_report()
print(summary)

In [ ]:
# Save report
report_path = Path('../reports/pay_equity_analysis.txt')
report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, 'w') as f:
    f.write(summary)

print(f"✓ Report saved to {report_path}")

In [ ]:
# Key takeaways
print("\n" + "=" * 60)
print("KEY FINDINGS")
print("=" * 60)

print(f"""
1. RAW WAGE GAP: {raw_gap['raw_gap']['mean_gap_pct']:.1f}%
   Women earn ${raw_gap['raw_gap']['female_to_male_ratio']:.2f} for every $1 men earn

2. STATISTICAL SIGNIFICANCE: 
   The wage gap is statistically significant (p < 0.001)
   Effect size: {raw_gap['statistical_tests']['effect_size']['cohens_d']:.3f} ({raw_gap['statistical_tests']['effect_size']['interpretation']})
""")

if 'decomp' in dir():
    print(f"""3. OAXACA-BLINDER DECOMPOSITION:
   - Explained by characteristics: {decomp['explained']['pct_of_total']:.0f}% of gap
   - Unexplained (potential bias): {decomp['unexplained']['pct_of_total']:.0f}% of gap
""")

print(f"""4. GLASS CEILING ANALYSIS: 
   {quantile_results['glass_ceiling_effect']['interpretation']}

5. POLICY IMPLICATIONS:
   Organizations should review pay practices to ensure compliance
   with Canadian Pay Equity Act requirements.
""")

## 1. Load Data

In [ ]:
# Import LFSDataPipeline if needed for data regeneration
from src.data_pipeline import LFSDataPipeline

# Load REAL Statistics Canada data
data_path = Path('../data/processed/lfs_processed.csv')

if data_path.exists():
    df = pd.read_csv(data_path)
    # Verify data source
    if 'source' in df.columns:
        source = df['source'].iloc[0] if len(df) > 0 else 'Unknown'
        print(f"✓ Data source: {source}")
        if 'StatCan_API_Real' not in str(source):
            print("⚠️ Data is not from real StatCan API. Regenerating...")
            pipeline = LFSDataPipeline()
            df = pipeline.generate_synthetic_data(n_samples=10000)
            df = pipeline.clean_data(df)
            df = pipeline.create_derived_features(df)
            df.to_csv(data_path, index=False)
else:
    print("Fetching REAL data from Statistics Canada API...")
    pipeline = LFSDataPipeline()
    df = pipeline.generate_synthetic_data(n_samples=10000)
    df = pipeline.clean_data(df)
    df = pipeline.create_derived_features(df)
    data_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(data_path, index=False)

print(f"✓ Dataset: {len(df):,} records")

# Gender breakdown (use dynamic column name)
gender_col = COLS.GENDER if COLS.GENDER in df.columns else 'SEX'
print(f"\nGender distribution:")
print(f"  Male: {(df[gender_col] == 1).sum():,} ({(df[gender_col] == 1).mean()*100:.1f}%)")
print(f"  Female: {(df[gender_col] == 2).sum():,} ({(df[gender_col] == 2).mean()*100:.1f}%)")

## 2. Raw Wage Gap Analysis

In [ ]:
# Initialize analyzer
analyzer = PayEquityAnalyzer(df)

# Compute raw wage gap
raw_gap = analyzer.compute_raw_wage_gap()

# Display results
print("="*60)
print("RAW GENDER WAGE GAP ANALYSIS")
print("="*60)

print(f"\nMale Workers:")
print(f"  Mean hourly wage: ${raw_gap['male']['mean']:.2f}")
print(f"  Median hourly wage: ${raw_gap['male']['median']:.2f}")
print(f"  Sample size: {raw_gap['male']['n']:,}")

print(f"\nFemale Workers:")
print(f"  Mean hourly wage: ${raw_gap['female']['mean']:.2f}")
print(f"  Median hourly wage: ${raw_gap['female']['median']:.2f}")
print(f"  Sample size: {raw_gap['female']['n']:,}")

print(f"\n" + "-"*60)
print(f"RAW WAGE GAP: {raw_gap['raw_gap']['mean_gap_pct']:.1f}%")
print(f"Women earn ${raw_gap['raw_gap']['female_to_male_ratio']:.2f} for every $1.00 men earn")
print(f"Dollar difference: ${raw_gap['raw_gap']['mean_gap']:.2f}/hour")

In [ ]:
# Statistical significance
print("\nStatistical Significance:")
print("-"*40)
t_test = raw_gap['statistical_tests']['t_test']
print(f"T-statistic: {t_test['statistic']:.2f}")
print(f"P-value: {t_test['p_value']:.2e}")
print(f"Significant at α=0.05: {'Yes ✓' if t_test['significant_05'] else 'No'}")
print(f"Significant at α=0.01: {'Yes ✓' if t_test['significant_01'] else 'No'}")

effect = raw_gap['statistical_tests']['effect_size']
print(f"\nEffect Size (Cohen's d): {effect['cohens_d']:.3f} ({effect['interpretation']})")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Bar comparison
ax = axes[0]
genders = ['Male', 'Female']
means = [raw_gap['male']['mean'], raw_gap['female']['mean']]
colors = ['#1f77b4', '#e377c2']
bars = ax.bar(genders, means, color=colors)
ax.bar_label(bars, fmt='$%.2f')
ax.set_ylabel('Average Hourly Wage ($)')
ax.set_title('Average Wage by Gender')

# Box plots
ax = axes[1]
df['Gender'] = df[gender_col].map({1: 'Male', 2: 'Female'})
df.boxplot(column='HRLYEARN', by='Gender', ax=ax)
ax.set_title('Wage Distribution by Gender')
ax.set_ylabel('Hourly Wage ($)')
plt.suptitle('')

# Gap visualization
ax = axes[2]
ax.barh(['Gap'], [raw_gap['raw_gap']['mean_gap_pct']], color='crimson')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Wage Gap (%)')
ax.set_title(f"Gender Wage Gap: {raw_gap['raw_gap']['mean_gap_pct']:.1f}%")
ax.set_xlim(0, max(20, raw_gap['raw_gap']['mean_gap_pct'] + 5))

plt.tight_layout()
plt.show()

## 3. Adjusted Wage Gap (Controlling for Factors)

In [ ]:
# Compute adjusted gap
# Control variables: education, experience, full-time status
control_vars = []
for col in ['EDUC', 'EXPERIENCE_PROXY', 'AGE_APPROX', 'FTPTMAIN']:
    if col in df.columns and df[col].dtype in ['int64', 'float64']:
        control_vars.append(col)

print(f"Control variables: {control_vars}")

if control_vars:
    adjusted_gap = analyzer.compute_adjusted_wage_gap(control_vars)
    
    print("\n" + "="*60)
    print("ADJUSTED WAGE GAP ANALYSIS")
    print("="*60)
    
    print(f"\nUnadjusted Model (gender only):")
    print(f"  Wage gap: {adjusted_gap['unadjusted_model']['gap_pct']:.1f}%")
    print(f"  R²: {adjusted_gap['unadjusted_model']['r_squared']:.4f}")
    
    print(f"\nAdjusted Model (with controls):")
    print(f"  Wage gap: {adjusted_gap['adjusted_model']['gap_pct']:.1f}%")
    print(f"  R²: {adjusted_gap['adjusted_model']['r_squared']:.4f}")
    print(f"  P-value: {adjusted_gap['adjusted_model']['p_value']:.4f}")
    
    print(f"\n" + "-"*60)
    print(f"Gap explained by controls: {adjusted_gap['gap_reduction']['absolute']:.1f} percentage points")
    print(f"Unexplained gap: {adjusted_gap['interpretation']['unexplained_gap']:.1f}%")
else:
    print("Insufficient control variables for adjusted analysis.")

In [ ]:
# Gap comparison visualization
if 'adjusted_gap' in dir():
    fig, ax = plt.subplots(figsize=(10, 5))
    
    gaps = [
        abs(adjusted_gap['unadjusted_model']['gap_pct']),
        abs(adjusted_gap['adjusted_model']['gap_pct'])
    ]
    labels = ['Unadjusted\n(Raw Gap)', 'Adjusted\n(Controlling for factors)']
    colors = ['#d62728', '#2ca02c']
    
    bars = ax.bar(labels, gaps, color=colors)
    ax.bar_label(bars, fmt='%.1f%%')
    ax.set_ylabel('Wage Gap (%)')
    ax.set_title('Raw vs Adjusted Gender Wage Gap')
    
    # Add annotation
    reduction = gaps[0] - gaps[1]
    ax.annotate(f'Reduction: {reduction:.1f}%', 
                xy=(0.5, max(gaps)/2), fontsize=12,
                ha='center', color='gray')
    
    plt.tight_layout()
    plt.show()

## 4. Oaxaca-Blinder Decomposition

In [ ]:
# Perform decomposition
if control_vars:
    decomp = analyzer.oaxaca_blinder_decomposition(control_vars)
    
    print("="*60)
    print("OAXACA-BLINDER WAGE DECOMPOSITION")
    print("="*60)
    
    print(f"\nTotal Wage Gap: {decomp['total_gap']['percentage']:.1f}%")
    
    print(f"\nExplained Component: {decomp['explained']['pct_of_total']:.1f}% of gap")
    print(f"  → {decomp['explained']['interpretation']}")
    
    print(f"\nUnexplained Component: {decomp['unexplained']['pct_of_total']:.1f}% of gap")
    print(f"  → {decomp['unexplained']['interpretation']}")
    
    print(f"\nSample sizes: Male={decomp['sample_sizes']['male']:,}, Female={decomp['sample_sizes']['female']:,}")

In [ ]:
# Decomposition visualization
if 'decomp' in dir():
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Pie chart of gap components
    sizes = [abs(decomp['explained']['pct_of_total']), 
             abs(decomp['unexplained']['pct_of_total'])]
    labels = ['Explained\n(Characteristics)', 'Unexplained\n(Potential Discrimination)']
    colors = ['#2ca02c', '#d62728']
    explode = (0, 0.05)
    
    ax.pie(sizes, explode=explode, labels=labels, colors=colors,
           autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
    ax.set_title(f'Wage Gap Decomposition\n(Total Gap: {decomp["total_gap"]["percentage"]:.1f}%)')
    
    plt.tight_layout()
    plt.show()

## 4.5 Macroeconomic Controls (Scientific Standards)

Adding macroeconomic controls (unemployment, GDP growth, inflation) accounts for business cycle effects on wages and meets publication standards for econometric analysis.

In [ ]:
# Import macro data module
import sys
sys.path.insert(0, str(Path.cwd().parent))

try:
    from src.macro_data import MACRO_DATA, get_macro_dataframe, ECONOMIC_PERIODS, get_macro_controls_summary
    MACRO_AVAILABLE = True
    print("✅ Macroeconomic data module loaded")
except ImportError:
    MACRO_AVAILABLE = False
    print("⚠️ Macro data module not available")

if MACRO_AVAILABLE:
    # Display macro data summary
    print("\n" + "="*60)
    print("MACROECONOMIC CONTROLS")
    print("="*60)
    
    macro_df = get_macro_dataframe()
    print("\nCanadian Economic Indicators (2010-2024):")
    display(macro_df[['year', 'cpi', 'gdp_growth', 'unemployment', 'interest_rate']].round(2))
    
    print("\nEconomic Periods:")
    for period, (start, end) in ECONOMIC_PERIODS.items():
        print(f"  {period.replace('_', ' ').title()}: {start}-{end}")

In [ ]:
# Compute macro-adjusted wage gap
if MACRO_AVAILABLE and 'YEAR' in df.columns and control_vars:
    macro_gap = analyzer.compute_macro_adjusted_wage_gap(control_vars, year_col='YEAR')
    
    print("="*60)
    print("MACRO-ADJUSTED WAGE GAP ANALYSIS")
    print("="*60)
    
    print(f"\nStandard Controls Only:")
    print(f"  Wage gap: {macro_gap['standard_controls']['gap_pct']:.1f}%")
    print(f"  R²: {macro_gap['standard_controls']['r_squared']:.4f}")
    
    print(f"\nWith Macroeconomic Controls:")
    print(f"  Wage gap: {macro_gap['macro_controls']['gap_pct']:.1f}%")
    print(f"  R²: {macro_gap['macro_controls']['r_squared']:.4f}")
    print(f"  Macro variables: {macro_gap['macro_controls']['macro_variables']}")
    
    print(f"\n" + "-"*60)
    print(f"Effect of adding macro controls: {macro_gap['macro_effect']['gap_change']:.2f} pp")
    
    # Show macro coefficients
    if 'macro_coefficients' in macro_gap:
        print("\nMacroeconomic Variable Effects:")
        for var, stats in macro_gap['macro_coefficients'].items():
            sig = "***" if stats['significant'] else ""
            print(f"  {var}: {stats['coefficient']:.4f} (p={stats['p_value']:.4f}){sig}")
elif not MACRO_AVAILABLE:
    print("⚠️ Macro module not available - skipping macro-adjusted analysis")
elif 'YEAR' not in df.columns:
    print("⚠️ YEAR column not in dataset - skipping macro-adjusted analysis")
else:
    print("⚠️ Control variables not available - skipping macro-adjusted analysis")

In [ ]:
# Visualize macro-adjusted vs standard gap
if 'macro_gap' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar comparison
    ax = axes[0]
    gaps = [
        abs(adjusted_gap['unadjusted_model']['gap_pct']),
        abs(adjusted_gap['adjusted_model']['gap_pct']),
        abs(macro_gap['macro_controls']['gap_pct'])
    ]
    labels = ['Raw Gap', 'Standard\nControls', 'With Macro\nControls']
    colors = ['#d62728', '#ff7f0e', '#2ca02c']
    
    bars = ax.bar(labels, gaps, color=colors)
    ax.bar_label(bars, fmt='%.1f%%')
    ax.set_ylabel('Wage Gap (%)')
    ax.set_title('Progressive Wage Gap Adjustment')
    
    # Macro coefficients
    ax = axes[1]
    if 'macro_coefficients' in macro_gap:
        vars_names = list(macro_gap['macro_coefficients'].keys())
        coeffs = [macro_gap['macro_coefficients'][v]['coefficient'] for v in vars_names]
        pvals = [macro_gap['macro_coefficients'][v]['p_value'] for v in vars_names]
        
        colors = ['#2ca02c' if p < 0.05 else '#d3d3d3' for p in pvals]
        bars = ax.barh(vars_names, coeffs, color=colors)
        ax.axvline(x=0, color='black', linewidth=0.5)
        ax.set_xlabel('Coefficient (effect on log wage)')
        ax.set_title('Macroeconomic Variable Effects\n(green = significant at 5%)')
    
    plt.tight_layout()
    plt.show()

## 5. Wage Gap by Demographic Groups

In [ ]:
# Labels
educ_labels = {
    0: 'Less than HS', 1: 'HS Graduate', 2: 'Some College',
    3: 'College Diploma', 4: 'University Cert', 5: "Bachelor's", 6: 'Graduate'
}

occ_labels = {
    0: 'Management', 1: 'Business', 2: 'Sciences', 3: 'Health',
    4: 'Education/Law', 5: 'Arts', 6: 'Sales/Service',
    7: 'Trades', 8: 'Resources', 9: 'Manufacturing'
}

# Gap by education
if 'EDUC' in df.columns:
    gap_by_educ = analyzer.analyze_by_group('EDUC')
    gap_by_educ['Education'] = gap_by_educ['EDUC'].map(educ_labels)
    
    print("Gender Wage Gap by Education Level:")
    display(gap_by_educ[['Education', 'male_mean', 'female_mean', 'gap_pct', 'significant']].round(2))

In [ ]:
# Visualize gap by education
if 'gap_by_educ' in dir():
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Drop rows where Education mapping failed (NaN values)
    gap_by_educ_clean = gap_by_educ.dropna(subset=['Education']).copy()
    gap_by_educ_sorted = gap_by_educ_clean.sort_values('gap_pct', ascending=True)
    
    colors = ['#d62728' if g > 10 else '#ffc107' if g > 5 else '#2ca02c' 
              for g in gap_by_educ_sorted['gap_pct']]
    
    ax.barh(gap_by_educ_sorted['Education'].astype(str), gap_by_educ_sorted['gap_pct'], color=colors)
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.set_xlabel('Wage Gap (%)')
    ax.set_title('Gender Wage Gap by Education Level')
    
    for i, (idx, row) in enumerate(gap_by_educ_sorted.iterrows()):
        ax.text(row['gap_pct'] + 0.3, i, f"{row['gap_pct']:.1f}%", va='center')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Gap by occupation
if 'NOC_10' in df.columns:
    gap_by_occ = analyzer.analyze_by_group('NOC_10')
    gap_by_occ['Occupation'] = gap_by_occ['NOC_10'].map(occ_labels)
    
    print("\nGender Wage Gap by Occupation:")
    display(gap_by_occ[['Occupation', 'male_mean', 'female_mean', 'gap_pct', 'significant']].round(2))

In [ ]:
# Visualize gap by occupation
if 'gap_by_occ' in dir():
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Drop rows where Occupation mapping failed (NaN values)
    gap_by_occ_clean = gap_by_occ.dropna(subset=['Occupation']).copy()
    gap_by_occ_sorted = gap_by_occ_clean.sort_values('gap_pct', ascending=True)
    
    colors = ['#d62728' if g > 15 else '#ffc107' if g > 10 else '#2ca02c' 
              for g in gap_by_occ_sorted['gap_pct']]
    
    ax.barh(gap_by_occ_sorted['Occupation'].astype(str), gap_by_occ_sorted['gap_pct'], color=colors)
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.set_xlabel('Wage Gap (%)')
    ax.set_title('Gender Wage Gap by Occupation Category')
    
    plt.tight_layout()
    plt.show()

## 6. Quantile Analysis (Glass Ceiling)

In [ ]:
# Quantile analysis
quantile_results = analyzer.quantile_analysis()

print("="*60)
print("WAGE GAP ACROSS WAGE DISTRIBUTION")
print("="*60)

for percentile, data in quantile_results.items():
    if percentile.startswith('p'):
        print(f"\n{percentile.upper()}:")
        print(f"  Male: ${data['male']:.2f}")
        print(f"  Female: ${data['female']:.2f}")
        print(f"  Gap: {data['gap_pct']:.1f}%")

print(f"\n" + "-"*60)
ceiling = quantile_results['glass_ceiling_effect']
print(f"Glass Ceiling Effect: {ceiling['interpretation']}")
print(f"Gap difference (P90 vs P10): {ceiling['difference']:.1f} percentage points")

In [ ]:
# Visualize quantile gaps
percentiles = ['p10', 'p25', 'p50', 'p75', 'p90']
gaps = [quantile_results[p]['gap_pct'] for p in percentiles]

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(percentiles, gaps, marker='o', linewidth=2, markersize=10)
ax.fill_between(percentiles, gaps, alpha=0.3)
ax.set_xlabel('Wage Percentile')
ax.set_ylabel('Wage Gap (%)')
ax.set_title('Gender Wage Gap Across Wage Distribution')
ax.set_xticklabels(['10th', '25th', '50th', '75th', '90th'])

for i, (p, g) in enumerate(zip(percentiles, gaps)):
    ax.annotate(f'{g:.1f}%', (p, g), textcoords='offset points', 
                xytext=(0, 10), ha='center')

plt.tight_layout()
plt.show()

## 7. Summary Report

In [ ]:
# Generate summary report
summary = analyzer.generate_summary_report()
print(summary)

In [ ]:
# Save report
report_path = Path('../reports/pay_equity_analysis.txt')
report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, 'w') as f:
    f.write(summary)

print(f"Report saved to {report_path}")

In [ ]:
# Key takeaways
print("\n" + "="*60)
print("KEY TAKEAWAYS FOR PORTFOLIO")
print("="*60)

print(f"""
1. RAW WAGE GAP: {raw_gap['raw_gap']['mean_gap_pct']:.1f}%
   Women earn ${raw_gap['raw_gap']['female_to_male_ratio']:.2f} for every $1 men earn

2. STATISTICAL SIGNIFICANCE: 
   The wage gap is statistically significant (p < 0.001)

3. DECOMPOSITION:
   - Explained by characteristics: {decomp['explained']['pct_of_total']:.0f}% of gap
   - Unexplained (potential bias): {decomp['unexplained']['pct_of_total']:.0f}% of gap

4. GLASS CEILING: 
   {quantile_results['glass_ceiling_effect']['interpretation']}

5. COMPLIANCE IMPLICATIONS:
   Organizations should review pay practices to ensure compliance
   with Canadian Pay Equity Act requirements.
""")